In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Step 1: Generate Embeddings
def generate_embeddings(data, model_name='all-mpnet-base-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(data, convert_to_tensor=False, normalize_embeddings=True)
    return embeddings

# Sample dataset
data = [
    "How to bake a cake?",
    "What is the capital of France?",
    "Tips for learning Python programming",
    "How to train a dog?",
    "Nifty 50 is falling",
    "Sensex is performing poorly",
    "Gold rates are rising",
    "Vegetables need sunlight",
    "Best practices for SEO optimization",
    "Carrots are tasty",
    "Photosynthesis process explained",
    "Dogs are loyal animals",
    "Be kind to animals",
    "Stock market downturn",
    "The fish market has a strong smell",
    "How to start a vegetable garden?",
    "Understanding the stock market basics",
    "Benefits of regular exercise",
    "How to make homemade pizza?",
    "History of the Roman Empire",
]

# Generate embeddings
embeddings = generate_embeddings(data)

# Step 2: Set up FAISS index
d = embeddings.shape[1]  # Dimension of embeddings
index = faiss.IndexFlatIP(d)  # Cosine similarity search
index.add(np.array(embeddings))

# Step 3: Insert Data into the Database
print(f"Number of vectors in the index: {index.ntotal}")

# Step 4: Query the Database
queries = [
    "How do I grow vegetables at home?",
    "What is the best way to train a puppy?",
    "What is the stock market?"
]

relevant_docs_list = [
    {7, 10, 15},  # Relevant docs for first query
    {3, 11},  # Relevant docs for second query
    {4, 5, 13, 16}  # Relevant docs for third query
]

k_values = [1, 3, 5]  # Different k values to test

def evaluate_metrics(query, relevant_docs, k):
    query_embedding = generate_embeddings([query])[0]
    distances, indices = index.search(np.array([query_embedding]), k)
    retrieved_docs = indices[0]

    precision_at_k = len(set(retrieved_docs) & relevant_docs) / k
    recall_at_k = len(set(retrieved_docs) & relevant_docs) / len(relevant_docs)

    def mean_reciprocal_rank(relevant_docs, retrieved_docs):
        for i, doc in enumerate(retrieved_docs, start=1):
            if doc in relevant_docs:
                return 1 / i
        return 0

    mrr = mean_reciprocal_rank(relevant_docs, retrieved_docs)

    def dcg(retrieved_docs, relevant_docs):
        return sum((1 / np.log2(i + 2)) if doc in relevant_docs else 0 for i, doc in enumerate(retrieved_docs))

    def ndcg_at_k(retrieved_docs, relevant_docs, k):
        ideal_dcg = dcg(sorted(relevant_docs, reverse=True), relevant_docs)
        actual_dcg = dcg(retrieved_docs[:k], relevant_docs)
        return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0

    ndcg = ndcg_at_k(retrieved_docs, relevant_docs, k)

    print(f"\nResults for k={k}")
    print("Query:", query)
    print("Top matches:")
    for i, idx in enumerate(retrieved_docs):
        print(f"{i + 1}. {data[idx]} (Score: {distances[0][i]:.4f})")

    print("\nEvaluation Metrics:")
    print(f"Precision@{k}: {precision_at_k:.4f}")
    print(f"Recall@{k}: {recall_at_k:.4f}")
    print(f"MRR: {mrr:.4f}")
    print(f"NDCG@{k}: {ndcg:.4f}")
    print("----------------------------")

for query, relevant_docs in zip(queries, relevant_docs_list):
    for k in k_values:
        evaluate_metrics(query, relevant_docs, k)


Number of vectors in the index: 20

Results for k=1
Query: How do I grow vegetables at home?
Top matches:
1. How to start a vegetable garden? (Score: 0.7359)

Evaluation Metrics:
Precision@1: 1.0000
Recall@1: 0.3333
MRR: 1.0000
NDCG@1: 0.4693
----------------------------

Results for k=3
Query: How do I grow vegetables at home?
Top matches:
1. How to start a vegetable garden? (Score: 0.7359)
2. Vegetables need sunlight (Score: 0.4713)
3. How to make homemade pizza? (Score: 0.4075)

Evaluation Metrics:
Precision@3: 0.6667
Recall@3: 0.6667
MRR: 1.0000
NDCG@3: 0.7654
----------------------------

Results for k=5
Query: How do I grow vegetables at home?
Top matches:
1. How to start a vegetable garden? (Score: 0.7359)
2. Vegetables need sunlight (Score: 0.4713)
3. How to make homemade pizza? (Score: 0.4075)
4. Carrots are tasty (Score: 0.3283)
5. How to bake a cake? (Score: 0.2435)

Evaluation Metrics:
Precision@5: 0.4000
Recall@5: 0.6667
MRR: 1.0000
NDCG@5: 0.7654
-------------------------